<a href="https://colab.research.google.com/github/priyal6/AI-AGENT/blob/main/agenteval_deterministic_trial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install the missing package
!pip install langchain_openai

from typing import TypedDict, List
from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI
from langchain.tools import tool
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage

In [ ]:
class AgentState(TypedDict):
  messages: List[BaseMessage]
  tool_calls: List[str]

In [ ]:
@tool
def get_weather(city: str) -> str:
  """Get the weather for a city"""
  return f"The weather in {city} is sunny."

In [ ]:
from google.colab import userdata
import os

os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
)

In [ ]:
def agent_node(state: AgentState):
  response = llm.invoke(state['messages'])

  return {
      "messages": state["messages"] + [response],
      "tool_calls": state["tool_calls"]
  }

In [ ]:
def tool_node(state: AgentState):
  last_msg = state["messages"][-1]

  if "weather" in last_msg.content.lower():
    tool_result = get_weather.invoke({'city':'Paris'})
    state['tool_calls'].append("get_weather")

    return {
        "messages": state["messages"] + [
            AIMessage(content=tool_result)
            ],
        "tool_calls": state["tool_calls"],
    }

  return state

In [ ]:
graph = StateGraph(AgentState)
graph.add_node("agent", agent_node)
graph.add_node("tool", tool_node)

graph.set_entry_point("agent")

graph.add_edge("agent","tool")
graph.add_edge("tool", END)

app = graph.compile()

In [ ]:
#dry run the agent

def run_agent(user_input:str):
  state = {
      "messages": [HumanMessage(content=user_input)],
      "tool_calls": [],
  }

  result = app.invoke(state)
  return result

In [ ]:
result = run_agent("What's the weather in Paris?")
print(result)

{'messages': [HumanMessage(content="What's the weather in Paris?", additional_kwargs={}, response_metadata={}), AIMessage(content="I'm unable to provide real-time weather updates. For the current weather in Paris, I recommend checking a reliable weather website or app.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 13, 'total_tokens': 39, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_f4ae844694', 'id': 'chatcmpl-D6nHheB4CqMPlydgsKsNMxMh3Bk3a', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019c3ab9-eb8e-7400-8037-a0c3b9b3a868-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 13, 'output_tokens': 26, 'total_tok

In [ ]:
#eval task

class EvalTask(TypedDict):
  name: str
  input: str
  allowed_tools: List[str]
  must_call_tool: bool

In [ ]:
#binary grade

def grade(task: EvalTask, result) -> bool:
  tool_calls = result["tool_calls"]

  for tool in tool_calls:
    if tool not in task["allowed_tools"]:
      return False

  if task["must_call_tool"] and not tool_calls:
    return False

  return True

In [ ]:
def run_eval(task: EvalTask, trials=5):
    results = []

    for _ in range(trials):
      result = run_agent(task["input"])
      success = grade(task, result)
      results.append(success)

    pass_at_k = 1.0 if any(results) else 0.0
    pass_pow_k = 1.0 if all(results) else 0.0

    return{
        "task": task["name"],
        "pass@k": pass_at_k,
        "pass^k": pass_pow_k,
        "raw": results,
    }


In [ ]:
eval_result = run_eval(weather_task, trials=5)
print(eval_result)

{'task': 'weather_query', 'pass@k': 1.0, 'pass^k': 1.0, 'raw': [True, True, True, True, True]}
